In [1]:
from fvcore.nn import FlopCountAnalysis
import torch

# ── Load Sparse Model ─────────────────────────────────────────────
from model import *  # import from your model.py

# ← add this!
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sparse_model = SparseAutoencoder(in_channels=8).to(device)
checkpoint = torch.load("models/sparse_autoencoder_checkpoint_2.pth", map_location=device)
if 'model_state_dict' in checkpoint:
    sparse_model.load_state_dict(checkpoint['model_state_dict'],strict = False)
else:
    sparse_model.load_state_dict(checkpoint)
sparse_model.eval()
print("Sparse model loaded ✅")

# ── Load Dense Model ──────────────────────────────────────────────
dense_model = DenseAutoencoder(in_channels=8).to(device)
checkpoint = torch.load("models/dense.pth", map_location=device)
if 'model_state_dict' in checkpoint:
    dense_model.load_state_dict(checkpoint['model_state_dict'])
else:
    dense_model.load_state_dict(checkpoint)
dense_model.eval()
print("Dense model loaded ✅")

Sparse model loaded ✅
Dense model loaded ✅


In [2]:
dataset   = H5Dataset("dataset/Dataset_Specific_Unlabelled.h5", x_key="jet")
loader    = DataLoader(dataset, batch_size=100, shuffle=True,
                       num_workers=0, pin_memory=True)

In [3]:
from fvcore.nn import FlopCountAnalysis

# FLOPs comparison
dummy_x    = torch.zeros(1, 8, 125, 125).to(device)
dummy_mask = torch.ones(1, 1, 125, 125).bool().to(device)

# sparse model flops
flops_sparse = FlopCountAnalysis(sparse_model, (dummy_x, dummy_mask))
print(f"Sparse FLOPs: {flops_sparse.total() / 1e6:.1f} M")

# dense model flops
flops_dense = FlopCountAnalysis(dense_model, dummy_x)
print(f"Dense FLOPs:  {flops_dense.total() / 1e6:.1f} M")

# active hidden states
sparse_model.eval()
with torch.no_grad():
    X, mask = next(iter(loader))
    X, mask = X.to(device), mask.to(device)
    _, latent_sparse = sparse_model(X, mask)
    _, latent_dense  = dense_model(X)

sparse_active = (latent_sparse.abs() > 0).sum().item()
dense_active  = latent_dense.numel()

print(f"Sparse active states: {sparse_active / 1000:.1f}K")
print(f"Dense active states:  {dense_active  / 1000:.1f}K")
print(f"Reduction: {dense_active / sparse_active:.1f}x")


Unsupported operator aten::mul encountered 14 time(s)
Unsupported operator aten::sub encountered 5 time(s)
Unsupported operator aten::adaptive_max_pool2d encountered 2 time(s)


Sparse FLOPs: 2069.4 M
Dense FLOPs:  1259.0 M
Sparse active states: 6513.3K
Dense active states:  13107.2K
Reduction: 2.0x
